# Pengambilan dan Pemahaman Data



1.   Variabel target: Sales
2.   Makna variabel target: total omzet atau pendapatan penjualan harian dari sebuah toko Rossmann pada tanggal ternetu

1.   Jumlah data observasi di train.csv: 1.017.209 baris data
2.   Nama kolom waktu atau tanggal: Date

1.   Variabel selain var target: Store, DayOfWeek, Date, Sales, Customers, Open, Promo, StateHoliday, SchoolHoliday
2.   Makna var:
      *   Open: Status operasional toko (1 = Buka, 0 = Tutup)
      *   Promo: Status promo di toko (1 = Ya, 0 = Tidak)
      *   StateHoliday: indikator hari libur negara bagian/nasional ('a' = public holiday, 'b' = Easter holiday, 'c' = Christmas, '0' = None)
      *   SchoolHoliday: sekolah libur atau tidak (1 = Libur, 0 = Tidak libur)
1.   Missing value: tidak ada
2.   Data penjualan harian menunjukkan pola tren, fluktuasi, atau musiman: YA. Flaktuasi: naik turun penjualan (promo dan open), musiman (weekday weekend)












## Pra-Pemrosesan Data

In [ ]:
# ==========================================
# 1. LIBRARY MANIPULASI DATA & SISTEM
# ==========================================
import os
import warnings
import pandas as pd
import numpy as np

# Mengabaikan semua pesan warning
warnings.filterwarnings('ignore')

# ==========================================
# 2. LIBRARY VISUALISASI DATA (EDA)
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Setup tema visualisasi agar clean dan estetik
sns.set_theme(style="whitegrid", palette="muted")

# ==========================================
# 3. LIBRARY MODEL STATISTIK KLASIK (SARIMA)
# ==========================================
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

# ==========================================
# 4. LIBRARY SCALING & EVALUASI METRIK
# ==========================================
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

# ==========================================
# 5. LIBRARY DEEP LEARNING (TensorFlow / Keras)
# ==========================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping

# Cek apakah GPU Kaggle sudah aktif untuk TensorFlow
print("TensorFlow Version:", tf.__version__)
print("GPU Tersedia:", tf.config.list_physical_devices('GPU'))

In [ ]:
# 1. Load Data
df = pd.read_csv('/kaggle/input/competitions/rossmann-store-sales/train.csv', low_memory=False)
store = pd.read_csv('/kaggle/input/competitions/rossmann-store-sales/store.csv')
# --- Pra-pemrosesan Data Sesuai Alur Logika ---

# e. Memilih satu toko tertentu (Store = 1) agar menjadi deret waktu tunggal
df_store1 = df[df['Store'] == 1].copy()

# g. Menghapus data ketika toko tutup (Open = 0)
df_store1 = df_store1[df_store1['Open'] == 1].copy()

# a. Mengubah kolom tanggal menjadi format datetime
df_store1['Date'] = pd.to_datetime(df_store1['Date'])

# c. Mengurutkan data berdasarkan waktu (paling awal ke akhir)
df_store1 = df_store1.sort_values('Date')

# b. Menjadikan kolom tanggal sebagai index data
df_store1.set_index('Date', inplace=True)

# d. Memeriksa dan menangani missing value jika ditemukan (Forward Fill)
df_store1 = df_store1.ffill()

# f. Memilih variabel Sales sebagai target peramalan
df_sales = df_store1[['Sales']]

# h. Membagi data menjadi data latih dan data uji (proporsi 80:20)
train_size = int(len(df_sales) * 0.8)
train_data = df_sales.iloc[:train_size]
test_data = df_sales.iloc[train_size:]

# i. Melakukan normalisasi data khusus untuk model RNN, LSTM, dan GRU
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

print(f"Pra-pemrosesan Selesai!\nJumlah Data Latih: {len(train_data)} baris\nJumlah Data Uji: {len(test_data)} baris")

## Eksplorasi Data Time Series

In [ ]:
# Setup tema visualisasi yang clean
sns.set_theme(style="whitegrid", palette="muted")

# Ekstraksi fitur tambahan untuk visualisasi
df_store1['Month'] = df_store1.index.month
df_store1['DayOfWeek'] = df_store1.index.dayofweek # 0=Senin, 6=Minggu

# Buat figure grid untuk menghemat tempat dan terlihat rapi
fig = plt.figure(figsize=(18, 16))

# a. Grafik penjualan harian berdasarkan waktu
ax1 = plt.subplot(3, 2, (1, 2))
ax1.plot(df_sales.index, df_sales['Sales'], color='#2ca02c', linewidth=1)
ax1.set_title('a. Penjualan Harian Keseluruhan Toko 1 (2013-2015)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Sales')

# b. Grafik penjualan dalam periode tertentu (Misal: Jan-Feb 2014)
ax2 = plt.subplot(3, 2, 3)
df_subset = df_sales.loc['2014-01-01':'2014-02-28']
ax2.plot(df_subset.index, df_subset['Sales'], marker='o', markersize=4, color='#1f77b4')
ax2.set_title('b. Penjualan Harian (Jan - Feb 2014)', fontsize=12, fontweight='bold')
plt.xticks(rotation=45)

# c. Grafik rata-rata penjualan berdasarkan bulan
ax3 = plt.subplot(3, 2, 4)
sns.barplot(data=df_store1, x='Month', y='Sales', estimator=np.mean, errorbar=None, ax=ax3, color='#9467bd')
ax3.set_title('c. Rata-rata Penjualan per Bulan', fontsize=12, fontweight='bold')

# d. Grafik rata-rata penjualan berdasarkan hari dalam seminggu
ax4 = plt.subplot(3, 2, 5)
sns.barplot(data=df_store1, x='DayOfWeek', y='Sales', estimator=np.mean, errorbar=None, ax=ax4, color='#ff7f0e')
ax4.set_title('d. Rata-rata Penjualan per Hari (0=Senin, 6=Minggu)', fontsize=12, fontweight='bold')

# e. Grafik perbandingan promosi
ax5 = plt.subplot(3, 2, 6)
sns.boxplot(data=df_store1, x='Promo', y='Sales', ax=ax5, palette='pastel')
ax5.set_title('e. Penjualan Saat Promo (1) vs Tidak Promo (0)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# h & i. Plot ACF dan PACF
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(df_sales['Sales'], lags=30, ax=axes[0], color='#17becf')
axes[0].set_title('h. Autocorrelation Function (ACF)')
plot_pacf(df_sales['Sales'], lags=30, ax=axes[1], color='#e377c2')
axes[1].set_title('i. Partial Autocorrelation Function (PACF)')
plt.show()

Interpretasi Karakteristik Data Penjualan (Store 1):

Pola Musiman (Seasonality): Terdapat pola musiman mingguan yang sangat jelas. Penjualan melonjak di hari Senin (Hari ke-0) dan stabil menurun ke hari Sabtu. Hari Minggu tidak ada data karena toko tutup. Hal ini juga terkonfirmasi dari grafik ACF yang menunjukkan lonjakan korelasi setiap 6-7 hari (lags).

Dampak Promosi (Fluktuasi): Fluktuasi data penjualan sangat dipengaruhi oleh variabel promosi. Grafik boxplot menunjukkan nilai median penjualan saat hari promo (Promo = 1) jauh lebih tinggi dibandingkan saat tidak ada promo (Promo = 0).

Pola Tren (Trend): Jika dilihat secara keseluruhan (2013-2015), tren penjualan cenderung stabil mendatar (stasioner) tanpa ada kenaikan/penurunan jangka panjang yang drastis, kecuali lonjakan wajar pada periode akhir tahun.

## Langkah 4: Pemodelan Menggunakan SARIMA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

# --- a. Uji Stasioneritas menggunakan Augmented Dickey-Fuller Test ---
print("--- Hasil Augmented Dickey-Fuller (ADF) Test ---")
# Kita uji pada data latih
result_adf = adfuller(train_data['Sales'])
print(f'ADF Statistic: {result_adf[0]:.4f}')
print(f'p-value: {result_adf[1]:.4f}')

if result_adf[1] <= 0.05:
    print("Kesimpulan: Data sudah Stasioner (p-value <= 0.05)")
else:
    print("Kesimpulan: Data Tidak Stasioner (p-value > 0.05), perlu differencing.")

# --- b & c. Menentukan Parameter SARIMA (p,d,q)(P,D,Q,s) ---
# Sesuai instruksi soal, periode musiman s = 7 untuk pola mingguan
# Kita gunakan parameter standar untuk data ritel: (1,1,1)(1,1,1,7)
p, d, q = 1, 1, 1
P, D, Q, s = 1, 1, 1, 7

print(f"\nMelatih Model SARIMA ({p},{d},{q})({P},{D},{Q},{s})...")

# --- d. Melatih Model SARIMA menggunakan data latih ---
# Note: enforce_stationarity=False agar model lebih fleksibel saat training
model_sarima = SARIMAX(train_data['Sales'],
                       order=(p, d, q),
                       seasonal_order=(P, D, Q, s),
                       enforce_stationarity=False,
                       enforce_invertibility=False)

results_sarima = model_sarima.fit(disp=False)
print("Model SARIMA berhasil dilatih!")

# --- e. Melakukan Prediksi terhadap data uji ---
# Kita prediksi sebanyak jumlah data di test_data
forecast_sarima = results_sarima.get_forecast(steps=len(test_data))
pred_sarima = forecast_sarima.predicted_mean

# --- f. Membandingkan Hasil Prediksi dengan Data Aktual ---
plt.figure(figsize=(12, 6))
plt.plot(train_data.index[-30:], train_data['Sales'][-30:], label='Data Latih (30 Hari Terakhir)')
plt.plot(test_data.index, test_data['Sales'], label='Data Aktual (Test)', color='black')
plt.plot(test_data.index, pred_sarima, label='Prediksi SARIMA', color='red', linestyle='--')
plt.title('f. Perbandingan Data Aktual vs Prediksi SARIMA', fontsize=14, fontweight='bold')
plt.legend()
plt.show()

# Simpan metrik untuk tabel evaluasi akhir nanti
mae_s = mean_absolute_error(test_data['Sales'], pred_sarima)
rmse_s = np.sqrt(mean_squared_error(test_data['Sales'], pred_sarima))
mape_s = mean_absolute_percentage_error(test_data['Sales'], pred_sarima)
r2_s = r2_score(test_data['Sales'], pred_sarima)

print(f"\nMetrik Evaluasi SARIMA:")
print(f"MAE: {mae_s:.2f} | RMSE: {rmse_s:.2f} | MAPE: {mape_s:.4f} | R2: {r2_s:.4f}")

## Pemodelan Menggunakan RNN (Langkah 5)

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

# --- b. Membentuk data menjadi sequence (Timestep = 7 hari) ---
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length)])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

seq_length = 7

# --- c. Membagi data sequence menjadi data latih dan data uji ---
X_train, y_train = create_sequences(train_scaled, seq_length)
X_test, y_test = create_sequences(test_scaled, seq_length)

print(f"Bentuk X_train: {X_train.shape} (Jumlah data, Timestep, Fitur)")

# --- d. Membangun arsitektur model RNN sederhana ---
model_rnn = Sequential()
# Menggunakan 50 neuron, fungsi aktivasi relu
model_rnn.add(SimpleRNN(50, activation='relu', input_shape=(seq_length, 1)))
model_rnn.add(Dense(1)) # Output layer untuk memprediksi 1 nilai (Sales)

model_rnn.compile(optimizer='adam', loss='mse')

# --- e. Melatih model RNN ---
print("\nMelatih model RNN (tunggu sebentar ya...)...")
# Epochs = 50, batch_size = 16
history_rnn = model_rnn.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)
print("Model RNN berhasil dilatih!")

# --- f. Melakukan prediksi terhadap data uji ---
pred_rnn_scaled = model_rnn.predict(X_test)

# --- g. Mengembalikan hasil prediksi ke skala asli ---
# Ingat, hasil di atas masih berskala 0-1, wajib dikembalikan ke nilai Sales ribuan
pred_rnn = scaler.inverse_transform(pred_rnn_scaled)

# y_test juga dikembalikan ke skala asli untuk hitung error di evaluasi nanti
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

print("Prediksi selesai dan sudah dikembalikan ke skala asli!")

## Pemodelan Menggunakan LSTM (Langkah 6)

In [ ]:
print("Membangun dan melatih model LSTM...")

# b & c. Membangun arsitektur model LSTM
model_lstm = Sequential()
model_lstm.add(LSTM(50, activation='relu', input_shape=(seq_length, 1)))
model_lstm.add(Dense(1))
model_lstm.compile(optimizer='adam', loss='mse')

# d. Melatih model LSTM
model_lstm.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)
print("Model LSTM selesai dilatih!")

# e & f. Melakukan prediksi & Inverse Transform
pred_lstm_scaled = model_lstm.predict(X_test)
pred_lstm = scaler.inverse_transform(pred_lstm_scaled)

## Pemodelan Menggunakan GRU (Langkah 7)

In [ ]:
print("Membangun dan melatih model GRU...")

# b & c. Membangun arsitektur model GRU
model_gru = Sequential()
model_gru.add(GRU(50, activation='relu', input_shape=(seq_length, 1)))
model_gru.add(Dense(1))
model_gru.compile(optimizer='adam', loss='mse')

# d. Melatih model GRU
model_gru.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)
print("Model GRU selesai dilatih!")

# e & f. Melakukan prediksi & Inverse Transform
pred_gru_scaled = model_gru.predict(X_test)
pred_gru = scaler.inverse_transform(pred_gru_scaled)

## Evaluasi Model (Langkah 8)

In [ ]:
## from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
import pandas as pd

# Karena sequence memotong 7 hari pertama dari data uji,
# kita harus potong juga hasil prediksi SARIMA agar panjang datanya sama
aligned_pred_sarima = pred_sarima[seq_length:]

# Kumpulkan semua prediksi
models_eval = {
    'SARIMA': aligned_pred_sarima,
    'RNN': pred_rnn.flatten(),
    'LSTM': pred_lstm.flatten(),
    'GRU': pred_gru.flatten()
}

eval_results = []
for name, pred in models_eval.items():
    mae = mean_absolute_error(y_test_actual, pred)
    rmse = np.sqrt(mean_squared_error(y_test_actual, pred))
    mape = mean_absolute_percentage_error(y_test_actual, pred)
    r2 = r2_score(y_test_actual, pred)

    eval_results.append([name, round(mae, 2), round(rmse, 2), round(mape, 4), round(r2, 4)])

# Buat tabel evaluasi
df_eval = pd.DataFrame(eval_results, columns=['Model', 'MAE', 'RMSE', 'MAPE', 'R2 Score'])
print("--- TABEL EVALUASI KINERJA MODEL ---")
print(df_eval.to_string(index=False))

## Visualisasi Gabungan (Langkah 9)

In [ ]:
import matplotlib.pyplot as plt

# Mengambil index tanggal yang sesuai untuk sumbu X (karena terpotong 7 hari seq_length)
dates_test = test_data.index[seq_length:]

plt.figure(figsize=(16, 8))
plt.plot(dates_test, y_test_actual, label='Data Aktual (Sales)', color='black', linewidth=2.5)

# Plot prediksi masing-masing model
plt.plot(dates_test, aligned_pred_sarima, label='Prediksi SARIMA', linestyle='--', color='red')
plt.plot(dates_test, pred_rnn, label='Prediksi RNN', alpha=0.8, color='orange')
plt.plot(dates_test, pred_lstm, label='Prediksi LSTM', alpha=0.8, color='blue')
plt.plot(dates_test, pred_gru, label='Prediksi GRU', alpha=0.8, color='green')

plt.title('Grafik Gabungan Data Aktual vs Hasil Prediksi Seluruh Model (Store 1)', fontsize=16, fontweight='bold')
plt.xlabel('Tanggal')
plt.ylabel('Sales')
plt.legend(loc='best')
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()